In [8]:
import pandas as pd
import numpy as np
import os
from typing import Iterator, Union, List, Optional
from pathlib import Path
import shutil
import pytz  # ensure this is at the top of your script

root_folder = '../../../../'
planning_data = os.path.join(root_folder,'planning')
aware_folder = os.path.join(root_folder,'3_Analytics_Warehouse_Backend')

traffic_email_path = os.path.join(planning_data,'3_emails_from_traffic')
phase2_sensor_path = os.path.join(planning_data,'4_phase_2_sensors/P2')

column_names_location = os.path.join(aware_folder,'1_inputs')#'../../../1_inputs'
parquet_intermediate_location = os.path.join(aware_folder,'2_planning_processing_files')#'../../../2_planning_processing_files'

clear_folder = os.path.join(traffic_email_path, 'clear')

In [9]:
# Read the CSV and extract column names as a list
cols = pd.read_csv(os.path.join(column_names_location, "columns.csv"))["Column Names"].tolist()
cols_p2 = pd.read_csv(os.path.join(column_names_location, "p2_columns.csv"))["Column Names"].tolist()
veh_cols = pd.read_csv(os.path.join(column_names_location, "veh_cols.csv"))["Column Names"].tolist()

print(cols)  # Display the list of column names
print(cols_p2)  # Display the list of column names
print(veh_cols)

xlsx_files_p2 = [f for f in os.listdir(phase2_sensor_path) if f.endswith(('.xlsx', '.xlsm'))]
# print(xlsx_files_p2)

all_files = [f for f in os.listdir(traffic_email_path) if f.endswith(('.xlsx', '.xlsm', '.pdf'))]

# Separate by file type
excel_files = [f for f in all_files if f.lower().endswith(('.xlsx', '.xlsm'))]
pdf_files = [f for f in all_files if f.lower().endswith('.pdf')]

# Separate files based on naming
pedestrian_counts =  [f for f in excel_files if 'MONTHLY' in f.upper() or 'P1' in f.upper()]
veh_report_files = [f for f in excel_files if 'WTC_VEH_REPORT' in f.upper()]

other_excel_files = [
    f for f in excel_files
    if f not in pedestrian_counts and f not in veh_report_files
]

# Move all PDFs and non-matching Excel files to 'clear'
files_to_clear = pdf_files + other_excel_files

for f in files_to_clear:
    src = os.path.join(traffic_email_path, f)
    dst = os.path.join(clear_folder, f)
    shutil.move(src, dst)

# Print results
print("P1 files:", pedestrian_counts)
print("P2 files:", xlsx_files_p2)
print("Vehicle report files:", veh_report_files)
print("Moved to 'clear':", files_to_clear)

['From', 'To Time', 'Z01_T4-ChurchSt_RevDoor_IN', 'Z01_T4-ChurchSt_RevDoor_OUT', 'Z01_T4-ChurchSt_SwingDoor_IN', 'Z01_T4-ChurchSt_SwingDoor_OUT', 'Z02_T4-LibertySt_EastEsc46_IN', 'Z02_T4-LibertySt_EastEsc46_OUT', 'Z02_T4-LibertySt_WestEsc45_IN', 'Z02_T4-LibertySt_WestEsc45_OUT', 'Z02_T4-LibertySt_Stair_IN', 'Z02_T4-LibertySt_Stair_OUT', 'Z02_T4-LibertySt_Doors_IN', 'Z02_T4-LibertySt_Doors_OUT', 'Z03_T4-C1-StairEsc_StairElev_IN', 'Z03_T4-C1-StairEsc_StairElev_OUT', 'Z03_T4-C1-StairEsc_EastEsc46_IN', 'Z03_T4-C1-StairEsc_EastEsc46_OUT', 'Z03_T4-C1-StairEsc_WestEsc45_IN', 'Z03_T4-C1-StairEsc_WestEsc45_OUT', 'Z04_T4-SoConc-Entry_AllDoors_IN', 'Z04_T4-SoConc-Entry_AllDoors_OUT', 'Z05_WestProj-Street_NorthDoors_IN', 'Z05_WestProj-Street_NorthDoors_OUT', 'Z05_WestProj-Street_SouthDoors_IN', 'Z05_WestProj-Street_SouthDoors_OUT', 'Z06_WestProj-C1-StairEsc_NorthStair_IN', 'Z06_WestProj-C1-StairEsc_NorthStair_OUT', 'Z06_WestProj-C1-StairEsc_NorthEsc36_IN', 'Z06_WestProj-C1-StairEsc_NorthEsc36_OUT'

In [11]:
for file in pedestrian_counts:
    file_path = os.path.join(traffic_email_path, file)
    arch_path = os.path.join(traffic_email_path, 'processed',file)

    try:
        # Read the Excel file with specified columns
        df = pd.read_excel(file_path, sheet_name="Data", usecols=cols, engine="openpyxl")
        df = df[df["From"].notnull()]
        df = df[~df["From"].astype(str).str.contains("screenline list", case=False, na=False)]
        print(f"Successfully read: {file}")
        # Generate the parquet filename dynamically
        parquet_filename = f"{parquet_intermediate_location}/p1_sensors_parquet/{file.rsplit('.', 1)[0]}.parquet"        # Save as Parquet
        df.to_parquet(parquet_filename, engine="pyarrow", index=False)
        print(f"Parquet file saved: {parquet_filename}")
        # Move the original file to archive
        dest = shutil.move(file_path, arch_path) 

    except Exception as e:
        print(f"Error reading {file}: {e}")
else:
    print("No valid data to save.")


No valid data to save.


In [12]:
possible_sheets = ["5-Min Data", "Sheet1", "Data"]  # Add all possible sheet names here

for file in xlsx_files_p2:
    file_path = os.path.join(phase2_sensor_path, file)
    arch_path = os.path.join(phase2_sensor_path, 'processed',file)

    df = None
    for sheet in possible_sheets:
        try:
            df = pd.read_excel(file_path, sheet_name=sheet, usecols=cols_p2, engine="openpyxl")
            if df["From"].notnull().any():
                print(f"Successfully read '{file}' using sheet '{sheet}'")
                break  # Exit the loop if successfully read
        except Exception as e:
            continue  # Try next sheet

    if df is None or df["From"].notnull().sum() == 0:
        print(f"No valid data found in {file}")
        continue

    try:
        df = df[df["From"].notnull()]
        parquet_filename = f"{parquet_intermediate_location}/p2_sensors_parquet/{file.rsplit('.', 1)[0]}.parquet"
        df.to_parquet(parquet_filename, engine="pyarrow", index=False)
        print(f"Parquet file saved: {parquet_filename}")
        shutil.move(file_path, arch_path)
    except Exception as e:
        print(f"Error processing {file}: {e}")



No valid data found in WTC_Daily Archive P2_20250605_00-03-34.xlsx
Successfully read 'WTC_Daily Archive P2_20250607_00-03-01.xlsx' using sheet 'Data'
Parquet file saved: ../../../../3_Analytics_Warehouse_Backend\2_planning_processing_files/p2_sensors_parquet/WTC_Daily Archive P2_20250607_00-03-01.parquet
Successfully read 'WTC_Daily Archive P2_20250608_00-03-41.xlsx' using sheet 'Data'
Parquet file saved: ../../../../3_Analytics_Warehouse_Backend\2_planning_processing_files/p2_sensors_parquet/WTC_Daily Archive P2_20250608_00-03-41.parquet
Successfully read 'WTC_Daily Archive P2_20250609_00-08-25.xlsx' using sheet 'Data'
Parquet file saved: ../../../../3_Analytics_Warehouse_Backend\2_planning_processing_files/p2_sensors_parquet/WTC_Daily Archive P2_20250609_00-08-25.parquet


In [13]:
possible_sheets = ["Raw POV Data"]  # Add all possible sheet names here

for file in veh_report_files:
    file_path = os.path.join(traffic_email_path, file)
    arch_path = os.path.join(traffic_email_path, 'processed', file)

    df = None  # reset for each file
    for sheet in possible_sheets:
        try:
            df = pd.read_excel(file_path, sheet_name=sheet, usecols= veh_cols,engine="openpyxl")
            if "Ticket" in df.columns and df["Ticket"].notnull().any():
                print(f"Successfully read '{file}' using sheet '{sheet}'")
                break
        except Exception as e:
            continue  # Try next sheet if one fails
    if df is None or "Ticket" not in df.columns or df["Ticket"].notnull().sum() == 0:
        print(f"No valid data found in {file}")
        continue

    try:
        df = df[df["Ticket"].notnull()]

        # Step 1: Standardize column names
        df.columns = (
            df.columns.str.strip()
            .str.lower()
            .str.replace(" ", "_")
            .str.replace("/", "_")  # Replace slashes with underscore
            .str.replace(r"[^\w_]", "", regex=True)  # Optional: remove all other non-alphanum (except _)
        )
    # Ensure 'Vehicle Plate' is all strings (if it exists)
        if "vehicle_plate" in df.columns:
            df["vehicle_plate"] = df["vehicle_plate"].astype(str)

        # Make sure these match your actual column names after cleaning
        datetime_fields = {
            "arrived_at": "arrive",
            "grant_deny_at": "grant",
            "vsc_entry": "vsc_entry",
            "gantry_exit": "gantry_exit"
        }

        for source_col, base_name in datetime_fields.items():
            dt_series = pd.to_datetime(df[source_col], errors='coerce', utc=True)
            # dt_series = dt_series.dt.tz_convert('US/Eastern').dt.tz_localize(None)

            df[f"{base_name}"] = dt_series
            df[f"{base_name}_date"] = dt_series.dt.date
            df[f"{base_name}_time"] = dt_series.dt.time
            df[f"{base_name}_rounded"] = dt_series.dt.floor("30min").dt.time

        # Base columns to keep
        base_cols = [
            "ticket", "vehicle_plate", "vehicle_plate_state", "vehicle_type",
            "vendor_tenant_association", "driver_name", "driver_license_state",
            "control_point_name", "ticket_notes", "notes"
        ]

        # Dynamically generated columns
        generated_cols = []
        for _, base in datetime_fields.items():
            generated_cols += [
                f"{base}_date", f"{base}_time", f"{base}_rounded"
            ]

        # Final selection — only keep relevant columns
        keep_cols = base_cols + generated_cols
        df = df[keep_cols]

        parquet_path = os.path.join(
            parquet_intermediate_location, "veh_parquet", f"{file.rsplit('.', 1)[0]}.parquet"
        )
        # os.makedirs(os.path.dirname(parquet_path), exist_ok=True)

        df.to_parquet(parquet_path, engine="pyarrow", index=False)
        print(f"Parquet file saved: {parquet_path}")
        shutil.move(file_path, arch_path)

    except Exception as e:
        print(f"Error processing {file}: {e}")


In [6]:
df

,From,To Time,Z08_HEC1M-ELEV18_IN,Z08_HEC1M-ELEV18_OUT,Z33_C1-WestConcourse_IN,Z33_C1-WestConcourse_OUT,Z32_T1ESC19&20/ZONE32_NorthEsc_IN,Z32_T1ESC19&20/ZONE32_NorthEsc_OUT,Z32_T1ESC19&20/ZONE32_SouththEsc_IN,Z32_T1ESC19&20/ZONE32_SouththEsc_OUT,...,Z19_PATHturnstiles-East_249-255(south)_IN,Z19_PATHturnstiles-East_249-255(south)_OUT,Z19_PATHturnstiles-East_256-261(center)_IN,Z19_PATHturnstiles-East_256-261(center)_OUT,Z19_PATHturnstiles-East_262-267(center)_IN,Z19_PATHturnstiles-East_262-267(center)_OUT,Z19_PATHturnstiles-East_268-274(north)_IN,Z19_PATHturnstiles-East_268-274(north)_OUT,Z20_PATHturnstiles-SE_Passageway-North_NB,Z20_PATHturnstiles-SE_Passageway-North_SB
0,2025-06-05 00:00:00,2025-06-05 00:05:00,0,1,0,0,0,0,0,0,...,0,7,0,1,0,1,0,1,2,1
1,2025-06-05 00:05:00,2025-06-05 00:10:00,0,0,0,0,0,0,0,0,...,3,10,0,8,0,0,0,5,5,2
2,2025-06-05 00:10:00,2025-06-05 00:15:00,0,0,0,0,0,0,0,0,...,0,29,0,8,0,2,0,6,0,1
3,2025-06-05 00:15:00,2025-06-05 00:20:00,0,1,0,0,0,0,0,0,...,0,16,0,2,0,1,0,5,0,1
4,2025-06-05 00:20:00,2025-06-05 00:25:00,0,0,0,0,0,0,0,0,...,12,34,0,13,0,9,11,2,0,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,2025-06-05 23:35:00,2025-06-05 23:40:00,0,0,0,0,0,0,0,0,...,0,20,0,6,0,2,1,5,2,1
284,2025-06-05 23:40:00,2025-06-05 23:45:00,0,7,0,0,0,0,2,0,...,41,42,6,18,0,12,40,11,3,4
285,2025-06-05 23:45:00,2025-06-05 23:50:00,0,2,0,0,0,0,3,0,...,7,35,0,9,0,10,6,6,1,7
286,2025-06-05 23:50:00,2025-06-05 23:55:00,1,0,0,0,0,0,4,0,...,0,34,0,12,0,6,0,4,0,4
